# Iterative Temporal Parent Discovery

Dev | Jacqueline Maasch | August 2026

In [1]:
# General imports.
import pandas as pd
import numpy as np
#import pickle
import matplotlib.pyplot as plt
import time
from string import ascii_uppercase as alphabet

# Time series causal discovery.
import tigramite
from tigramite import data_processing as pp
from tigramite.toymodels import structural_causal_processes as toys
from tigramite import plotting as tp
from tigramite.pcmci import PCMCI # also for PCMCI+
from tigramite.independence_tests.gsquared import Gsquared # univariate discrete/categorical vars
from tigramite.independence_tests.cmisymb import CMIsymb # multivariate discrete/categorical vars (permutation-based)
from tigramite.independence_tests.parcorr import ParCorr # univariate, continuous variables with linear dependencies and Gaussian noise

# VLCD.
from vlcd import PCMCIConfig, pcmci

# Vanilla causal discovery.
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
from causallearn.utils.cit import CIT

# Custom discovery algorithms.
from padl_itpd import PaDL
from itpd import ITPD, Utils
# Naive implementation.
from padl import PaDL
from itpd_naive import ITPDNaive

# Metrics.
#from dodiscover.metrics import structure_hamming_dist as SHD
from cdt.metrics import precision_recall
from cdt.metrics import SHD
from cdt.metrics import SID

# Graphical modeling.
import networkx as nx

No GPU automatically detected. Setting SETTINGS.GPU to 0, and SETTINGS.NJOBS to cpu_count.


# ITPD

## Generate random time series

In [14]:
# Get random data.
u = Utils()
dfs = []
graphs = []
adj_matrices = []
p2c_list = []
c2p_list = []
M = 1_0
T = 10
N = 10
max_children = 4
plot = False
iters = 10 # total random datasets to generate.

for i in range(iters):
    df, p2c, c2p, G, true_adj = u.get_data(M = M, 
                                           T = T,
                                           N = N, 
                                           max_children = max_children,
                                           plot = plot)
    dfs.append(df)
    graphs.append(G)
    adj_matrices.append(true_adj)
    p2c_list.append(p2c)
    c2p_list.append(c2p)

In [15]:
dfs[0]

,A_0,A_1,A_2,A_3,A_4,A_5,A_6,A_7,A_8,A_9,...,J_0,J_1,J_2,J_3,J_4,J_5,J_6,J_7,J_8,J_9
0,3.283953,-4.504600,-3.222043,-2.561838,-2.901063,3.310611,-1.441506,-4.198031,3.656297,2.247699,...,0.432200,0.941022,-1.137000,-0.035338,-3.186508,4.353372,-5.053277,-4.215821,-4.003840,26.449784
1,-0.529878,0.444776,0.538817,1.723158,1.852976,2.417034,-3.120846,-2.284285,3.277677,4.712836,...,-0.069303,0.204725,-0.121267,-6.042229,-9.159585,7.793611,-10.225854,-7.211851,-50.737557,74.808091
2,0.424182,-0.262188,-0.073994,0.261143,-0.649085,-1.485723,-0.329790,-1.551681,0.727614,4.117752,...,0.258888,0.380266,0.634551,2.112267,0.453073,5.489937,-6.805768,-6.372176,-18.423994,34.027003
3,0.567904,-1.004953,-0.722616,-1.956611,-3.061262,3.330790,-1.875997,-3.365471,0.485904,2.060546,...,-0.648502,-0.657739,-0.482801,-2.858571,-3.174339,2.942051,-3.020903,-6.911739,-5.878167,2.167498
4,-0.365017,0.878981,1.266262,-0.910261,-1.871113,4.975216,-2.544958,-1.517413,1.598449,-0.464015,...,-0.228595,1.053709,-3.420952,-8.379564,-8.542445,1.988769,-3.538402,-4.095749,16.072301,-9.431646
5,0.893413,-1.379279,-1.192483,-0.859043,-2.176499,-0.805962,3.019697,8.744166,-5.857021,-4.412709,...,1.855617,2.161423,-2.596046,-3.926181,-4.226856,6.400683,-11.170400,-9.263772,-6.149293,26.237267
6,0.261472,0.244641,-1.372745,0.596865,1.268516,-1.626776,4.889419,4.233649,-6.090949,0.208849,...,1.600487,1.701613,-1.207027,1.689075,0.153367,2.985664,-4.405138,-10.154918,-1.205981,-2.984988
7,-1.117703,2.030532,0.143074,1.436281,2.716140,-3.263591,1.252986,3.857514,-4.227278,-7.259035,...,-1.064476,-0.079306,0.054823,-1.847422,-1.706905,-0.561023,-0.143310,-2.164793,-33.874290,44.212756
8,0.296609,0.896819,0.626753,1.032138,1.037909,0.694814,1.463285,0.162924,-0.798887,-5.689843,...,1.327629,0.317019,1.040172,0.385021,4.473917,-4.455885,5.983754,4.143142,3.796496,-15.069692
9,-0.569156,-1.393361,0.140304,1.003860,1.759524,1.640108,0.105985,5.098319,-5.129511,-3.427917,...,0.769856,0.063705,-0.887053,-1.834709,-4.967957,1.194559,-0.953662,-3.600150,1.315433,2.850694


In [16]:
p2c_list[0]

{'A_0': ['A_1', 'J_8', 'E_3'],
 'A_1': ['A_2', 'B_9', 'I_9', 'B_5'],
 'A_2': ['A_3', 'H_5', 'I_6'],
 'A_3': ['A_4', 'F_8', 'B_8'],
 'A_4': ['A_5', 'D_8', 'F_8', 'G_5', 'E_6'],
 'A_5': ['A_6', 'J_9'],
 'A_6': ['A_7', 'I_8', 'B_8'],
 'A_7': ['A_8'],
 'A_8': ['A_9', 'F_9'],
 'A_9': [],
 'B_0': ['B_1', 'I_8'],
 'B_1': ['B_2', 'D_2', 'C_7', 'J_4', 'E_9'],
 'B_2': ['B_3', 'C_9', 'J_3', 'B_7'],
 'B_3': ['B_4'],
 'B_4': ['B_5', 'E_6'],
 'B_5': ['B_6', 'A_7', 'H_8', 'E_7'],
 'B_6': ['B_7'],
 'B_7': ['B_8'],
 'B_8': ['B_9', 'J_9', 'I_9', 'H_9', 'C_9'],
 'B_9': [],
 'C_0': ['C_1', 'A_8'],
 'C_1': ['C_2', 'J_7', 'C_6', 'H_2', 'H_4'],
 'C_2': ['C_3', 'E_5'],
 'C_3': ['C_4', 'A_6', 'E_5'],
 'C_4': ['C_5'],
 'C_5': ['C_6', 'I_8', 'A_8', 'C_8', 'E_8'],
 'C_6': ['C_7', 'F_8', 'J_8', 'C_8'],
 'C_7': ['C_8', 'I_9', 'J_8', 'D_9', 'G_9'],
 'C_8': ['C_9', 'A_9'],
 'C_9': [],
 'D_0': ['D_1', 'I_3', 'I_5', 'J_5'],
 'D_1': ['D_2', 'A_5', 'I_9'],
 'D_2': ['D_3'],
 'D_3': ['D_4'],
 'D_4': ['D_5', 'I_5', 'J_9', '

## Oracle

### ITPD

In [17]:
#%%capture

#'''
test = "oracle"
alpha = 0.01
tau_max = None
pairs = None
var_names = list(alphabet[:N])

accuracies = []
auprs = []
shds = []
total_tests = []
cond_sizes = []
itpd_oracle_times = []

for i in range(iters):
    start = time.time()
    it = ITPD(dfs[i],
              var_names = var_names,
              pairs = pairs,
              independence_test = test,
              alpha = alpha,
              tau_max = tau_max,
              true_adj = adj_matrices[i])
    accuracy, aupr, shd = it.itpd()
    itpd_oracle_times.append(time.time() - start)
    
    total_tests.append(np.sum(it.total_tests))
    #cond_sizes += it.conditioning_set_sizes
    
    accuracies.append(accuracy)
    auprs.append(aupr)
    shds.append(shd)

print()
print(f"Mean runtime     : {round(np.mean(itpd_oracle_times),4)} seconds")
print(f"Mean total tests : {round(np.mean(total_tests),2)}")
#print(f"Mean cond. set : {round(np.mean(cond_sizes),2)}")
print()
print(f"Mean accuracy    : {round(np.mean(accuracies)*100,2)}")
print(f"Mean AUPR        : {round(np.mean(auprs)*100,2)}")
print(f"Mean SHD         : {round(np.mean(shds),2)}")
#'''

ITPD complete in 2.8728 seconds.
ITPD complete in 3.0281 seconds.
ITPD complete in 3.0452 seconds.
ITPD complete in 3.1144 seconds.
ITPD complete in 2.8997 seconds.
ITPD complete in 2.8836 seconds.
ITPD complete in 2.9223 seconds.
ITPD complete in 3.0632 seconds.
ITPD complete in 2.9661 seconds.
ITPD complete in 2.9053 seconds.

Mean runtime     : 2.9714 seconds
Mean total tests : 16075.5

Mean accuracy    : 100.0
Mean AUPR        : 100.0
Mean SHD         : 0.0


### ITPD naive

In [18]:
naive_accuracies = []
naive_auprs = []
naive_shds = []
naive_total_tests = []
naive_cond_sizes = []
naive_itpd_oracle_times = []

for i in range(iters):
    start = time.time()
    it = ITPDNaive(dfs[i],
                  var_names = var_names,
                  pairs = pairs,
                  independence_test = test,
                  alpha = alpha,
                  tau_max = tau_max,
                  true_adj = adj_matrices[i])
    accuracy, aupr, shd = it.itpd_naive()
    naive_itpd_oracle_times.append(time.time() - start)
    
    naive_total_tests.append(np.sum(it.total_tests))
    #cond_sizes += it.conditioning_set_sizes
    
    naive_accuracies.append(accuracy)
    naive_auprs.append(aupr)
    naive_shds.append(shd)

print()
print(f"Mean runtime     : {round(np.mean(naive_itpd_oracle_times),4)} seconds")
print(f"Mean total tests : {round(np.mean(naive_total_tests),2)}")
#print(f"Mean cond. set : {round(np.mean(cond_sizes),2)}")
print()
print(f"Mean accuracy    : {round(np.mean(naive_accuracies)*100,2)}")
print(f"Mean AUPR        : {round(np.mean(naive_auprs)*100,2)}")
print(f"Mean SHD         : {round(np.mean(naive_shds),2)}")

ITPD complete in 3.2488 seconds.
ITPD complete in 3.4033 seconds.
ITPD complete in 3.3713 seconds.
ITPD complete in 3.678 seconds.
ITPD complete in 3.3409 seconds.
ITPD complete in 3.2821 seconds.
ITPD complete in 3.2896 seconds.
ITPD complete in 3.4865 seconds.
ITPD complete in 3.2245 seconds.
ITPD complete in 3.3395 seconds.

Mean runtime     : 3.3676 seconds
Mean total tests : 18107.7

Mean accuracy    : 100.0
Mean AUPR        : 100.0
Mean SHD         : 0.0


In [19]:
np.mean(itpd_oracle_times) / np.mean(naive_itpd_oracle_times)

np.float64(0.8823343544868364)

In [20]:
np.mean(total_tests) / np.mean(naive_total_tests)

np.float64(0.8877715005218774)

# End of document